# Jupiter Notebook for evaluating the dataset quality
## Part 2 : Basic sanity checks of the dataset


In [6]:
import json
from pathlib import Path
from collections import Counter, defaultdict
import statistics

# 1. LOAD FINAL DATASET

dataset_file = Path(
    r"C:\Users\mgdka\PycharmProjects\PythonProject\CREATING_DATA\final_dataset_cleaned.json"
)

with dataset_file.open("r", encoding="utf-8") as f:
    data = json.load(f)

print("Loaded examples:", len(data))


# 2. BASIC STRUCTURE CHECKS

required_fields = [
    "example_id",
    "owner",
    "original_contract_id",
    "question",
    "source_A",
    "source_B",
    "case_label",
    "answer_supported_by_A",
    "answer_supported_by_B",
    "expected_behaviour",
]

print("\n--- MISSING REQUIRED FIELDS ---")

for field in required_fields:
    missing = [
        item["example_id"]
        for item in data
        if item.get(field) is None
        or (isinstance(item.get(field), str) and not item[field].strip())
    ]

    print(f"{field}: {len(missing)} missing")

    if missing:
        print("  First examples:", missing[:10])


# 3. COUNTS BY OWNER AND CASE TYPE

print("\n--- EXAMPLES PER OWNER ---")
print(Counter(item["owner"] for item in data))

print("\n--- EXAMPLES PER CASE TYPE ---")
print(Counter(item["case_label"] for item in data))

print("\n--- CASE TYPES PER OWNER ---")

for owner in sorted(set(item["owner"] for item in data)):
    owner_rows = [item for item in data if item["owner"] == owner]
    print(owner, Counter(item["case_label"] for item in owner_rows))


# 4. CHECK UNIQUE EXAMPLE IDs

all_ids = [item["example_id"] for item in data]

duplicate_ids = [
    example_id
    for example_id, count in Counter(all_ids).items()
    if count > 1
]

print("\n--- EXAMPLE ID CHECK ---")
print("Unique IDs:", len(set(all_ids)))
print("Duplicate IDs:", duplicate_ids)


# 5. SOURCE TEXT LENGTHS

source_a_lengths = [
    len((item.get("source_A") or "").strip())
    for item in data
]

source_b_lengths = [
    len((item.get("source_B") or "").strip())
    for item in data
]

print("\n--- SOURCE TEXT LENGTHS ---")

print(
    "Source A | "
    f"min: {min(source_a_lengths)}, "
    f"median: {int(statistics.median(source_a_lengths))}, "
    f"max: {max(source_a_lengths)}"
)

print(
    "Source B | "
    f"min: {min(source_b_lengths)}, "
    f"median: {int(statistics.median(source_b_lengths))}, "
    f"max: {max(source_b_lengths)}"
)


# 6. FIND EMPTY OR SUSPICIOUSLY SHORT SOURCE TEXTS

THRESHOLD = 500

short_source_a = [
    item for item in data
    if len((item.get("source_A") or "").strip()) < THRESHOLD
]

short_source_b = [
    item for item in data
    if len((item.get("source_B") or "").strip()) < THRESHOLD
]

print(f"\n--- SOURCE TEXTS SHORTER THAN {THRESHOLD} CHARACTERS ---")
print("Short Source A:", len(short_source_a))
print("Short Source B:", len(short_source_b))

if short_source_a or short_source_b:
    print("\nSuspicious rows:")

    suspicious = {item["example_id"]: item for item in short_source_a + short_source_b}

    for item in suspicious.values():
        print(
            f"{item['example_id']} | {item['owner']} | "
            f"A={len((item.get('source_A') or '').strip())} | "
            f"B={len((item.get('source_B') or '').strip())} | "
            f"{item['original_contract_id']}"
        )


# 7. CHECK THE SOURCE A / SOURCE B RELATIONSHIP

print("\n--- SOURCE A / SOURCE B CONSISTENCY ---")

problems = []

for item in data:
    source_a = item["source_A"]
    source_b = item["source_B"]
    answer_a = item["answer_supported_by_A"].strip()
    answer_b = item["answer_supported_by_B"].strip()
    case = item["case_label"]

    texts_equal = source_a == source_b
    answers_equal = answer_a == answer_b

    # Expected logic for every experimental case:
    correct = (
        (case == "agreement" and texts_equal and answers_equal)
        or
        (case == "irrelevant_conflict" and not texts_equal and answers_equal)
        or
        (case == "relevant_conflict" and not texts_equal and not answers_equal)
    )

    if not correct:
        problems.append({
            "example_id": item["example_id"],
            "owner": item["owner"],
            "case_label": case,
            "texts_equal": texts_equal,
            "answers_equal": answers_equal,
            "question": item["question"],
        })

print("Inconsistent source/answer/case rows:", len(problems))

for problem in problems[:20]:
    print(problem)


# 8. CHECK SIX EXAMPLES PER ORIGINAL CONTRACT

contract_groups = defaultdict(list)

for item in data:
    key = (item["owner"], item["original_contract_id"])
    contract_groups[key].append(item)

group_problems = []

for (owner, contract_id), rows in contract_groups.items():
    case_counts = Counter(item["case_label"] for item in rows)

    expected_case_counts = {
        "agreement": 2,
        "irrelevant_conflict": 2,
        "relevant_conflict": 2,
    }

    if len(rows) != 6 or dict(case_counts) != expected_case_counts:
        group_problems.append({
            "owner": owner,
            "contract": contract_id,
            "number_of_rows": len(rows),
            "case_counts": dict(case_counts),
        })

print("\n--- CONTRACT GROUP CHECK ---")
print("Unique original contracts:", len(contract_groups))
print("Contracts not following the six-row design:", len(group_problems))

for problem in group_problems[:20]:
    print(problem)


# 9. CHECK CONTRACTS ARE NOT SHARED ACROSS MEMBERS

contract_owners = defaultdict(set)

for item in data:
    contract_owners[item["original_contract_id"]].add(item["owner"])

shared_contracts = {
    contract_id: owners
    for contract_id, owners in contract_owners.items()
    if len(owners) > 1
}

print("\n--- CONTRACT OVERLAP BETWEEN MEMBERS ---")
print("Contracts shared between members:", len(shared_contracts))

for contract_id, owners in list(shared_contracts.items())[:20]:
    print(contract_id, owners)


# 10. FINAL SUMMARY

print("\n================ FINAL SUMMARY ================")

if (
    not duplicate_ids
    and not short_source_a
    and not short_source_b
    and not problems
    and not group_problems
    and not shared_contracts
):
    print("PASSED: Dataset has no structural problems found.")
else:
    print("CHECK RESULTS: Review the issues printed above.")

Loaded examples: 324

--- MISSING REQUIRED FIELDS ---
example_id: 0 missing
owner: 0 missing
original_contract_id: 0 missing
question: 0 missing
source_A: 0 missing
source_B: 0 missing
case_label: 0 missing
answer_supported_by_A: 0 missing
answer_supported_by_B: 0 missing
expected_behaviour: 0 missing

--- EXAMPLES PER OWNER ---
Counter({'Oguz': 102, 'Magda': 60, 'Berkay': 60, 'Engin': 60, 'Dilek': 42})

--- EXAMPLES PER CASE TYPE ---
Counter({'agreement': 108, 'relevant_conflict': 108, 'irrelevant_conflict': 108})

--- CASE TYPES PER OWNER ---
Berkay Counter({'agreement': 20, 'relevant_conflict': 20, 'irrelevant_conflict': 20})
Dilek Counter({'agreement': 14, 'relevant_conflict': 14, 'irrelevant_conflict': 14})
Engin Counter({'agreement': 20, 'relevant_conflict': 20, 'irrelevant_conflict': 20})
Magda Counter({'agreement': 20, 'relevant_conflict': 20, 'irrelevant_conflict': 20})
Oguz Counter({'agreement': 34, 'relevant_conflict': 34, 'irrelevant_conflict': 34})

--- EXAMPLE ID CHECK --

In [7]:
from statistics import mean, median

# Change this path to your dataset file

# A Source B shorter/longer by more than this percentage gets flagged.
# 1% is strict; use 2% if you expect formatting differences.
MAX_PERCENT_DIFFERENCE = 1.0

DATASET_PATH = r"C:\Users\mgdka\PycharmProjects\PythonProject\CREATING_DATA\final_dataset_cleaned.json"

with open(DATASET_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)


def length_difference_percent(source_a, source_b):
    """Difference in character length, relative to Source A."""
    if len(source_a) == 0:
        return None

    return abs(len(source_b) - len(source_a)) / len(source_a) * 100


flagged_rows = []
all_differences = []

for row in data:
    source_a = row["source_A"]
    source_b = row["source_B"]

    len_a = len(source_a)
    len_b = len(source_b)
    difference = len_b - len_a
    difference_percent = length_difference_percent(source_a, source_b)

    all_differences.append(difference_percent)

    # Agreement rows should be completely identical
    if row["case_label"] == "agreement":
        if source_a != source_b:
            flagged_rows.append({
                "example_id": row["example_id"],
                "owner": row["owner"],
                "contract_id": row["original_contract_id"],
                "case_label": row["case_label"],
                "problem": "Agreement case: Source A and Source B are not identical.",
                "source_A_length": len_a,
                "source_B_length": len_b,
                "difference_characters": difference,
                "difference_percent": difference_percent,
            })

    # Conflict rows should stay almost the same length:
    # a minimal overwrite should normally only change a few characters.
    else:
        if difference_percent > MAX_PERCENT_DIFFERENCE:
            flagged_rows.append({
                "example_id": row["example_id"],
                "owner": row["owner"],
                "contract_id": row["original_contract_id"],
                "case_label": row["case_label"],
                "problem": (
                    f"Source B differs in length by more than "
                    f"{MAX_PERCENT_DIFFERENCE}% from Source A."
                ),
                "source_A_length": len_a,
                "source_B_length": len_b,
                "difference_characters": difference,
                "difference_percent": difference_percent,
            })


# ---------------- SUMMARY ----------------

print("=" * 100)
print("SOURCE LENGTH CHECK")
print("=" * 100)

print(f"Total dataset rows: {len(data)}")
print(f"Flagged rows: {len(flagged_rows)}")
print(f"Maximum allowed difference: {MAX_PERCENT_DIFFERENCE}%")

print("\nOverall length-difference statistics:")
print(f"Average difference: {mean(all_differences):.4f}%")
print(f"Median difference:  {median(all_differences):.4f}%")
print(f"Maximum difference: {max(all_differences):.4f}%")

print("\n" + "=" * 100)
print("FLAGGED ROWS")
print("=" * 100)

if not flagged_rows:
    print("No rows were flagged. All Source B texts are approximately the same length as Source A.")
else:
    for item in flagged_rows:
        print(f"\nExample ID: {item['example_id']}")
        print(f"Owner: {item['owner']}")
        print(f"Contract: {item['contract_id']}")
        print(f"Case label: {item['case_label']}")
        print(f"Problem: {item['problem']}")
        print(f"Source A length: {item['source_A_length']:,} characters")
        print(f"Source B length: {item['source_B_length']:,} characters")
        print(f"Difference: {item['difference_characters']:+,} characters")
        print(f"Difference percentage: {item['difference_percent']:.4f}%")

SOURCE LENGTH CHECK
Total dataset rows: 324
Flagged rows: 0
Maximum allowed difference: 1.0%

Overall length-difference statistics:
Average difference: 0.0210%
Median difference:  0.0016%
Maximum difference: 0.4019%

FLAGGED ROWS
No rows were flagged. All Source B texts are approximately the same length as Source A.


## Part 2 : Manuall check of 10 % of the dataset

This is where the manual check of the 10 percent of that dataset will take place.
We will check if the chages made in the contracts were applied correctly from the agent, as well as if they are concrete and correct


In [8]:
import random
import difflib
from collections import defaultdict

# ---------------- SETTINGS ----------------

N_CONTRACTS_TO_REVIEW = 5   # 5 × 6 = 30 examples = 9.3% of 324 rows
RANDOM_SEED = 42            # Keep this fixed so you get the same sample again
SHOW_FULL_SOURCE_TEXT = True # Set to False if you only want the differences
# ------------------------------------------

# Load dataset
with open(DATASET_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

# Group the six examples belonging to each original contract
contracts = defaultdict(list)

for row in data:
    key = (row["owner"], row["original_contract_id"])
    contracts[key].append(row)

# Keep only complete six-row contracts
complete_contracts = {
    key: rows
    for key, rows in contracts.items()
    if len(rows) == 6
}

print(f"Total rows: {len(data)}")
print(f"Complete contracts: {len(complete_contracts)}")
print(f"Reviewing: {N_CONTRACTS_TO_REVIEW} contracts / {N_CONTRACTS_TO_REVIEW * 6} rows")

# Randomly choose contracts
random.seed(RANDOM_SEED)
selected_keys = random.sample(
    list(complete_contracts.keys()),
    N_CONTRACTS_TO_REVIEW
)

# Put cases in the intended order
case_order = {
    "agreement": 0,
    "relevant_conflict": 1,
    "irrelevant_conflict": 2,
}

for number, key in enumerate(selected_keys, start=1):
    owner, contract_id = key
    rows = complete_contracts[key]

    # Sort to keep rows in their original dataset order
    rows = sorted(rows, key=lambda x: x["example_id"])

    print("\n" + "=" * 110)
    print(f"CONTRACT {number}/{N_CONTRACTS_TO_REVIEW}")
    print(f"Owner: {owner}")
    print(f"Contract ID: {contract_id}")
    print("=" * 110)

    # Source A is the same for all six rows
    source_a = rows[0]["source_A"]

    print("\nSOURCE A")
    print("-" * 110)
    if SHOW_FULL_SOURCE_TEXT:
        print(source_a)
    else:
        print(f"[Source A hidden: {len(source_a):,} characters]")

    # Group examples by Source B version.
    # In each contract there should be:
    # one B0 source (agreement), one B1 source, and one B2 source.
    source_b_groups = defaultdict(list)

    for row in rows:
        source_b_groups[row["source_B"]].append(row)

    for variant_number, (source_b, variant_rows) in enumerate(source_b_groups.items(), start=1):

        variant_rows = sorted(
            variant_rows,
            key=lambda x: (
                case_order.get(x["case_label"], 99),
                x["question"]
            )
        )

        print("\n" + "#" * 110)
        print(f"SOURCE B VARIANT {variant_number}")
        print("#" * 110)

        print("\nCASES USING THIS SOURCE B:")
        for row in variant_rows:
            print(f"\nExample ID: {row['example_id']}")
            print(f"Case label: {row['case_label']}")
            print(f"Question: {row['question']}")
            print(f"Answer supported by A: {row['answer_supported_by_A']}")
            print(f"Answer supported by B: {row['answer_supported_by_B']}")
            print(f"Agent's stated change: {row['exactly_what_changed']}")
            print(f"Expected behaviour: {row['expected_behaviour']}")

        # Print only the changed lines first: easiest way to validate the edit
        print("\nDIFFERENCE: SOURCE A → THIS SOURCE B")
        print("-" * 110)

        diff = difflib.unified_diff(
            source_a.splitlines(),
            source_b.splitlines(),
            fromfile="Source_A",
            tofile="Source_B",
            lineterm=""
        )

        diff_text = "\n".join(diff)

        if diff_text:
            print(diff_text)
        else:
            print("No difference: this should be an agreement case.")

        # Optionally print the full Source B as well
        if SHOW_FULL_SOURCE_TEXT:
            print("\nFULL SOURCE B")
            print("-" * 110)
            print(source_b)

print("\n" + "=" * 110)
print("Manual review finished.")

Total rows: 324
Complete contracts: 54
Reviewing: 5 contracts / 30 rows

CONTRACT 1/5
Owner: Oguz
Contract ID: HealthcareIntegratedTechnologiesInc_20190812_8-K_EX-10.1_11776966_EX-10.1_Reseller Agreement

SOURCE A
--------------------------------------------------------------------------------------------------------------
WALABOT-HOME RESELLER AGREEMENT This Walabot-HOME Reseller Agreement ("Agreement") is made and entered into as of this 31 day of July 2019 ("Effective Date") by and between Vayyar Imaging Ltd., having its principal place of business at 3 Avraham Giron St., POB. 325, Yehud 5621717, Israel ("Supplier"), and Inde Living Holdings, Inc., having its principal place of business at 1462 Rudder Lane, Knoxville, TN 37919 ("Reseller"). Vayyar and Reseller shall be referred to individually as "Party" and collectively as "Parties". 1. Definitions. The following capitalized terms shall have the following meanings: "Customer" means a third party who purchases the Products from Rese

# Part 3 : Checking the quality of the data with an agent


In [9]:
import json
import random
from pathlib import Path
from collections import defaultdict, Counter
from openai import OpenAI


# ==========================================================
# SETTINGS
# ==========================================================

DATASET_PATH = Path(
    r"C:\Users\mgdka\PycharmProjects\PythonProject\CREATING_DATA\final_dataset_cleaned.json"
)

REPORT_PATH = Path(
    r"C:\Users\mgdka\PycharmProjects\PythonProject\CREATING_DATA\quality_report\dataset_validation_report.json"
)

MODEL_NAME = "gpt-5"

# False = check every complete contract group (54 groups in the dataset) here the agent is checking ALL contract since this parameter is set to false
# True = check only a random sample first, to test the code/cost
USE_SAMPLE = False
N_CONTRACTS_TO_CHECK = 5
RANDOM_SEED = 42



# ==========================================================
# EXPECTED SIX-ROW DESIGN
# ==========================================================

EXPECTED_CASES = [
    ("B0", "Q1", "agreement"),
    ("B0", "Q2", "agreement"),
    ("B1", "Q1", "relevant_conflict"),
    ("B1", "Q2", "irrelevant_conflict"),
    ("B2", "Q1", "irrelevant_conflict"),
    ("B2", "Q2", "relevant_conflict"),
]


# ==========================================================
# AGENT OUTPUT SCHEMA
# ==========================================================

VALIDATION_SCHEMA = {
    "type": "object",
    "properties": {
        "overall_pass": {
            "type": "boolean",
            "description": "True only if the complete six-row design is valid."
        },
        "source_a_is_consistent": {
            "type": "boolean",
            "description": "Whether Source A is internally consistent enough for the questions."
        },
        "b1_relevant_conflict_valid": {
            "type": "boolean",
            "description": "Whether B1 genuinely changes the answer to Q1."
        },
        "b1_irrelevant_conflict_valid": {
            "type": "boolean",
            "description": "Whether B1 leaves Q2's answer unchanged despite changing another part."
        },
        "b2_relevant_conflict_valid": {
            "type": "boolean",
            "description": "Whether B2 genuinely changes the answer to Q2."
        },
        "b2_irrelevant_conflict_valid": {
            "type": "boolean",
            "description": "Whether B2 leaves Q1's answer unchanged despite changing another part."
        },
        "questions_are_clear_and_answerable": {
            "type": "boolean",
            "description": "Whether all questions can be answered directly from the supplied sources."
        },
        "changes_are_minimal_and_controlled": {
            "type": "boolean",
            "description": "Whether B1 and B2 make only the intended change, apart from necessary repeated references."
        },
        "issues": {
            "type": "array",
            "items": {"type": "string"},
            "description": "Concrete problems. Empty if no problems were found."
        },
        "recommended_action": {
            "type": "string",
            "enum": [
                "keep",
                "manual_review",
                "remove_or_regenerate"
            ]
        }
    },
    "required": [
        "overall_pass",
        "source_a_is_consistent",
        "b1_relevant_conflict_valid",
        "b1_irrelevant_conflict_valid",
        "b2_relevant_conflict_valid",
        "b2_irrelevant_conflict_valid",
        "questions_are_clear_and_answerable",
        "changes_are_minimal_and_controlled",
        "issues",
        "recommended_action"
    ],
    "additionalProperties": False
}


# ==========================================================
# DETERMINISTIC CHECKS
# These do not need an AI model.
# ==========================================================

def get_variant_and_question(example_id):
    """
    Extract B0/B1/B2 and Q1/Q2 only for Oguz IDs.
    Other owners have EX IDs, so we infer positions from
    the ordered six-row contract group instead.
    """
    parts = example_id.split("-")

    if len(parts) >= 4 and parts[-2] in {"B0", "B1", "B2"}:
        return parts[-2], parts[-1]

    return None, None


def deterministic_checks(rows):
    """
    Check the required structure and obvious consistency rules.
    """
    problems = []

    if len(rows) != 6:
        problems.append(
            f"Expected 6 rows for one contract, found {len(rows)}."
        )
        return {
            "passed": False,
            "problems": problems
        }

    # Dataset order should already be:
    # B0/Q1, B0/Q2, B1/Q1, B1/Q2, B2/Q1, B2/Q2
    expected_labels = [
        "agreement",
        "agreement",
        "relevant_conflict",
        "irrelevant_conflict",
        "irrelevant_conflict",
        "relevant_conflict",
    ]

    actual_labels = [row["case_label"] for row in rows]

    if actual_labels != expected_labels:
        problems.append(
            f"Incorrect case-label order.\n"
            f"Expected: {expected_labels}\n"
            f"Found:    {actual_labels}"
        )

    # All rows for a contract should use the same Source A
    source_a_values = {row["source_A"] for row in rows}

    if len(source_a_values) != 1:
        problems.append(
            "The six examples do not share exactly the same Source A."
        )

    source_a = rows[0]["source_A"]

    # Agreement cases: Source A must exactly equal Source B
    for row in rows[:2]:
        if row["source_A"] != row["source_B"]:
            problems.append(
                f"{row['example_id']}: Agreement case has different "
                f"Source A and Source B."
            )

        if row["answer_supported_by_A"].strip() != row["answer_supported_by_B"].strip():
            problems.append(
                f"{row['example_id']}: Agreement case has different "
                f"answers for Source A and Source B."
            )

    # Conflict cases: Source A and Source B must differ
    for row in rows[2:]:
        if row["source_A"] == row["source_B"]:
            problems.append(
                f"{row['example_id']}: Conflict case has identical "
                f"Source A and Source B."
            )

    # B1 rows must use the same Source B
    if rows[2]["source_B"] != rows[3]["source_B"]:
        problems.append(
            "B1/Q1 and B1/Q2 do not use the same Source B text."
        )

    # B2 rows must use the same Source B
    if rows[4]["source_B"] != rows[5]["source_B"]:
        problems.append(
            "B2/Q1 and B2/Q2 do not use the same Source B text."
        )

    # Relevant conflict: recorded answers must differ
    for row in [rows[2], rows[5]]:
        if row["answer_supported_by_A"].strip() == row["answer_supported_by_B"].strip():
            problems.append(
                f"{row['example_id']}: Relevant-conflict case has "
                f"the same recorded answer in A and B."
            )

    # Irrelevant conflict: recorded answers must stay the same
    for row in [rows[3], rows[4]]:
        if row["answer_supported_by_A"].strip() != row["answer_supported_by_B"].strip():
            problems.append(
                f"{row['example_id']}: Irrelevant-conflict case has "
                f"different recorded answers in A and B."
            )

    # Q1 should be identical in rows 0, 2, and 4
    q1_questions = {
        rows[0]["question"],
        rows[2]["question"],
        rows[4]["question"]
    }

    if len(q1_questions) != 1:
        problems.append(
            "The Q1 question is not identical across B0, B1, and B2."
        )

    # Q2 should be identical in rows 1, 3, and 5
    q2_questions = {
        rows[1]["question"],
        rows[3]["question"],
        rows[5]["question"]
    }

    if len(q2_questions) != 1:
        problems.append(
            "The Q2 question is not identical across B0, B1, and B2."
        )

    return {
        "passed": len(problems) == 0,
        "problems": problems
    }


# ==========================================================
# SEMANTIC AI VALIDATION
# ==========================================================

def ask_agent_to_validate(rows):
    """
    Give the model the full Source A, B1, B2 and the four meaningful
    conflict questions, so it can judge whether the edits make sense.
    """

    # Dataset order:
    # 0 = B0/Q1 agreement
    # 1 = B0/Q2 agreement
    # 2 = B1/Q1 relevant
    # 3 = B1/Q2 irrelevant
    # 4 = B2/Q1 irrelevant
    # 5 = B2/Q2 relevant

    q1 = rows[0]
    q2 = rows[1]
    b1_q1 = rows[2]
    b1_q2 = rows[3]
    b2_q1 = rows[4]
    b2_q2 = rows[5]

    prompt = f"""
You are validating a controlled legal two-source QA dataset.

Each contract should follow this intended design:

1. B0 agreement:
   - Source A and Source B are identical.
   - Q1 and Q2 have the same answer in both.

2. B1:
   - Only the answer to Q1 should materially change.
   - Q1 is therefore a relevant conflict.
   - Q2 should remain answerable with the same answer.
   - Q2 is therefore an irrelevant conflict.

3. B2:
   - Only the answer to Q2 should materially change.
   - Q2 is therefore a relevant conflict.
   - Q1 should remain answerable with the same answer.
   - Q1 is therefore an irrelevant conflict.

Check the source texts themselves, not only the stored labels or answers.
A change is acceptable only if it is legally meaningful, clearly supported by
the contract text, and does not accidentally alter an answer which should be
unchanged. Minor edits to repeated references are acceptable only when needed
to avoid an internal contradiction.

CONTRACT ID:
{q1["original_contract_id"]}

================ SOURCE A ================
{q1["source_A"]}

================ SOURCE B1 ================
{b1_q1["source_B"]}

================ SOURCE B2 ================
{b2_q1["source_B"]}

================ Q1 ================
Question: {q1["question"]}
Answer stated for Source A: {q1["answer_supported_by_A"]}
Answer stated for B1: {b1_q1["answer_supported_by_B"]}
Answer stated for B2: {b2_q1["answer_supported_by_B"]}
B1 claimed change: {b1_q1["exactly_what_changed"]}
B2 claimed change: {b2_q1["exactly_what_changed"]}

================ Q2 ================
Question: {q2["question"]}
Answer stated for Source A: {q2["answer_supported_by_A"]}
Answer stated for B1: {b1_q2["answer_supported_by_B"]}
Answer stated for B2: {b2_q2["answer_supported_by_B"]}
B1 claimed change: {b1_q2["exactly_what_changed"]}
B2 claimed change: {b2_q2["exactly_what_changed"]}

Return valid JSON only.
"""

    response = client.responses.create(
        model=MODEL_NAME,
        input=prompt,
        text={
            "format": {
                "type": "json_schema",
                "name": "contract_dataset_validation",
                "strict": True,
                "schema": VALIDATION_SCHEMA
            }
        },
        store=False
    )

    return json.loads(response.output_text)


# ==========================================================
# LOAD AND GROUP DATASET
# ==========================================================

with open(DATASET_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

contracts = defaultdict(list)

for row in data:
    key = (
        row["owner"],
        row["original_contract_id"]
    )
    contracts[key].append(row)

# Keep complete six-row contracts only
complete_contracts = {
    key: rows
    for key, rows in contracts.items()
    if len(rows) == 6
}

print(f"Total rows in dataset: {len(data)}")
print(f"Complete six-row contracts: {len(complete_contracts)}")

# Preserve original order in JSON file
for key in complete_contracts:
    complete_contracts[key] = sorted(
        complete_contracts[key],
        key=lambda row: data.index(row)
    )

selected_contracts = list(complete_contracts.items())

if USE_SAMPLE:
    random.seed(RANDOM_SEED)
    selected_contracts = random.sample(
        selected_contracts,
        min(N_CONTRACTS_TO_CHECK, len(selected_contracts))
    )

print(f"Contracts selected for AI validation: {len(selected_contracts)}")


# ==========================================================
# RUN VALIDATION
# ==========================================================

report = []

for number, ((owner, contract_id), rows) in enumerate(
    selected_contracts,
    start=1
):
    print("\n" + "=" * 100)
    print(f"Checking {number}/{len(selected_contracts)}")
    print(f"Owner: {owner}")
    print(f"Contract: {contract_id}")

    deterministic_result = deterministic_checks(rows)

    print(
        "Deterministic checks:",
        "PASS" if deterministic_result["passed"] else "FAIL"
    )

    try:
        semantic_result = ask_agent_to_validate(rows)

        print(
            "AI semantic validation:",
            "PASS" if semantic_result["overall_pass"] else "FLAGGED"
        )

    except Exception as error:
        semantic_result = {
            "overall_pass": False,
            "issues": [f"AI validation failed: {str(error)}"],
            "recommended_action": "manual_review"
        }

        print("AI semantic validation: ERROR")

    final_status = (
        "PASS"
        if deterministic_result["passed"]
        and semantic_result["overall_pass"]
        else "FLAGGED"
    )

    report.append({
        "owner": owner,
        "original_contract_id": contract_id,
        "final_status": final_status,
        "deterministic_checks": deterministic_result,
        "ai_semantic_validation": semantic_result,
        "example_ids": [row["example_id"] for row in rows]
    })

    # Save progress after every contract, in case credits run out
    with open(REPORT_PATH, "w", encoding="utf-8") as f:
        json.dump(report, f, ensure_ascii=False, indent=2)


# ==========================================================
# SUMMARY
# ==========================================================

summary = Counter(item["final_status"] for item in report)

print("\n" + "=" * 100)
print("FINAL SUMMARY")
print("=" * 100)
print(f"Checked contracts: {len(report)}")
print(f"PASS: {summary['PASS']}")
print(f"FLAGGED: {summary['FLAGGED']}")
print(f"\nSaved report to:\n{REPORT_PATH}")

print("\nFLAGGED CONTRACTS:")
for item in report:
    if item["final_status"] == "FLAGGED":
        print(f"\n- {item['owner']} | {item['original_contract_id']}")

        for problem in item["deterministic_checks"]["problems"]:
            print(f"  Deterministic issue: {problem}")

        for issue in item["ai_semantic_validation"].get("issues", []):
            print(f"  AI issue: {issue}")

Total rows in dataset: 324
Complete six-row contracts: 54
Contracts selected for AI validation: 54

Checking 1/54
Owner: Magda
Contract: ADAMSGOLFINC_03_21_2005-EX-10.17-ENDORSEMENT AGREEMENT
Deterministic checks: PASS
AI semantic validation: PASS

Checking 2/54
Owner: Magda
Contract: VEONEER,INC_02_21_2020-EX-10.11-JOINT VENTURE AGREEMENT
Deterministic checks: PASS
AI semantic validation: PASS

Checking 3/54
Owner: Magda
Contract: FTENETWORKS,INC_02_18_2016-EX-99.4-STRATEGIC ALLIANCE AGREEMENT
Deterministic checks: PASS
AI semantic validation: PASS

Checking 4/54
Owner: Magda
Contract: SIBANNAC,INC_12_04_2017-EX-2.1-Strategic Alliance Agreement
Deterministic checks: PASS
AI semantic validation: PASS

Checking 5/54
Owner: Magda
Contract: INTRICONCORP_03_10_2009-EX-10.22-Strategic Alliance Agreement
Deterministic checks: PASS
AI semantic validation: PASS

Checking 6/54
Owner: Magda
Contract: BORROWMONEYCOM,INC_06_11_2020-EX-10.1-JOINT VENTURE AGREEMENT
Deterministic checks: PASS
AI sema

## Remove the flagged contracts

In [10]:
import json
from collections import Counter

INPUT_PATH = r"C:\Users\mgdka\PycharmProjects\PythonProject\CREATING_DATA\final_dataset_cleaned.json"

OUTPUT_PATH = r"C:\Users\mgdka\PycharmProjects\PythonProject\CREATING_DATA\final_dataset_final.json"

CONTRACTS_TO_REMOVE = {
    "AIRTECHINTERNATIONALGROUPINC_05_08_2000-EX-10.4-FRANCHISE AGREEMENT",
    "CytodynInc_20200109_10-Q_EX-10.5_11941634_EX-10.5_License Agreement",
}

# Load dataset
with open(INPUT_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

# Check that each flagged contract has exactly six rows before deletion
rows_to_remove = [
    row for row in data
    if row["original_contract_id"] in CONTRACTS_TO_REMOVE
]

counts_to_remove = Counter(
    row["original_contract_id"] for row in rows_to_remove
)

print("Rows found for removal:")
for contract_id in CONTRACTS_TO_REMOVE:
    print(f"{contract_id}: {counts_to_remove[contract_id]} rows")

assert all(
    counts_to_remove[contract_id] == 6
    for contract_id in CONTRACTS_TO_REMOVE
), "Stop: each flagged contract should have exactly 6 rows."

# Keep all rows except the two flagged contract groups
cleaned_data = [
    row for row in data
    if row["original_contract_id"] not in CONTRACTS_TO_REMOVE
]

# Save new final dataset
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(cleaned_data, f, ensure_ascii=False, indent=2)

# Final checks
print("\nFINAL CHECK")
print("Original rows:", len(data))
print("Removed rows:", len(data) - len(cleaned_data))
print("Remaining rows:", len(cleaned_data))

print("\nExamples per case:")
print(Counter(row["case_label"] for row in cleaned_data))

remaining_contracts = {
    row["original_contract_id"] for row in cleaned_data
}

print("\nRemaining contracts:", len(remaining_contracts))

assert len(cleaned_data) == 312
assert len(remaining_contracts) == 52
assert not any(
    row["original_contract_id"] in CONTRACTS_TO_REMOVE
    for row in cleaned_data
)

print(f"\nSaved cleaned dataset to:\n{OUTPUT_PATH}")

Rows found for removal:
CytodynInc_20200109_10-Q_EX-10.5_11941634_EX-10.5_License Agreement: 6 rows
AIRTECHINTERNATIONALGROUPINC_05_08_2000-EX-10.4-FRANCHISE AGREEMENT: 6 rows

FINAL CHECK
Original rows: 324
Removed rows: 12
Remaining rows: 312

Examples per case:
Counter({'agreement': 104, 'relevant_conflict': 104, 'irrelevant_conflict': 104})

Remaining contracts: 52

Saved cleaned dataset to:
C:\Users\mgdka\PycharmProjects\PythonProject\CREATING_DATA\final_dataset_final.json
